# Предподготовка данных

## 1. Загрузка и первичный осмотр данных

In [1]:
# устанавливаем первые библиотеки
!pip install gdown -q

import gdown
import pandas as pd
import numpy as np
import re

In [2]:
#file_id = '17MzyZfT-s8vLl7s0ww3xJ0awR8-TG0aS'

#gdown.download(f'https://drive.google.com/uc?id={file_id}', 'data.csv', quiet=False)
#после первой загрузки оставить закомментированным

In [3]:
df = pd.read_csv('data.csv', sep = ';', decimal = ',')
display(df.head())

,Год,Месяц,Компания,Бренд,Модель,Модификация,Год выпуска,Страна-производитель,Вид топлива,"Объём двиг, л,",...,Тип клиента,Форма расчета,Количество,"Цена, USD","Продажа, USD",Область,Сегментация 2013,Класс 2013,Сегментация Eng,Локализация производства
0,2019,Май,Mercur Auto,Audi,A3,TFSI,2018,Германия,Бензин,"1,4",...,Физ. Лицо,безналичный,1.0,28115.00,28115.00,г.Алматы,Легковые автомобили,C класс,C,Импорт
1,2019,Август,Mercur Auto,Audi,A3,TFSI,2018,Германия,Бензин,"1,4",...,Юр. Лицо,наличный,1.0,32246.99,32246.99,г.Алматы,Легковые автомобили,C класс,C,Импорт
2,2019,Апрель,Mercur Auto,Audi,A4,TFSI,2018,Германия,Бензин,"1,4",...,Физ. Лицо,безналичный,1.0,32000.00,32000.00,г.Алматы,Легковые автомобили,D класс,D,Импорт
3,2019,Июль,Mercur Auto,Audi,A4,TFSI,2018,Германия,Бензин,"1,4",...,Юр. Лицо,безналичный,1.0,31929.00,31929.00,г.Алматы,Легковые автомобили,D класс,D,Импорт
4,2019,Июль,Mercur Auto,Audi,A4,TFSI,2018,Германия,Бензин,"1,4",...,Физ. Лицо,наличный,1.0,31929.00,31929.00,г.Алматы,Легковые автомобили,D класс,D,Импорт


In [4]:
# выводим общую информацию о датасете
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39966 entries, 0 to 39965
Data columns (total 25 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Год                             39966 non-null  int64  
 1   Месяц                           39966 non-null  object 
 2   Компания                        39966 non-null  object 
 3   Бренд                           39966 non-null  object 
 4   Модель                          39966 non-null  object 
 5   Модификация                     36375 non-null  object 
 6   Год выпуска                     39465 non-null  object 
 7   Страна-производитель            39966 non-null  object 
 8   Вид топлива                     36826 non-null  object 
 9   Объём двиг, л,                  35708 non-null  object 
 10  Коробка передач                 36711 non-null  object 
 11  Тип привода                     35677 non-null  object 
 12  Сегмент                         

видим, что типы данных соответствуют значениям

In [5]:
# выводим суммарное число пропусков
df.isna().sum()

Год                                   0
Месяц                                 0
Компания                              0
Бренд                                 0
Модель                                0
Модификация                        3591
Год выпуска                         501
Страна-производитель                  0
Вид топлива                        3140
Объём двиг, л,                     4258
Коробка передач                    3255
Тип привода                        4289
Сегмент                            6761
Регион                                0
Наименование дилерского центра        0
Тип клиента                        7047
Форма расчета                     25928
Количество                            6
Цена, USD                             0
Продажа, USD                          0
Область                               0
Сегментация 2013                      0
Класс 2013                            0
Сегментация Eng                       0
Локализация производства              0


да, пропусков у нас местами много, будем подправлять
удалим столбцы, содержащие критическое количество пропусков или не енсущие в себе пользы
для дальнейшего анализа. 

In [6]:
col_drop = [ 'Компания', 'Форма расчета', 'Сегмент', 'Сегментация Eng', 'Локализация производства', 'Тип клиента', 'Модификация']
df.drop(columns = col_drop , inplace = True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39966 entries, 0 to 39965
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Год                             39966 non-null  int64  
 1   Месяц                           39966 non-null  object 
 2   Бренд                           39966 non-null  object 
 3   Модель                          39966 non-null  object 
 4   Год выпуска                     39465 non-null  object 
 5   Страна-производитель            39966 non-null  object 
 6   Вид топлива                     36826 non-null  object 
 7   Объём двиг, л,                  35708 non-null  object 
 8   Коробка передач                 36711 non-null  object 
 9   Тип привода                     35677 non-null  object 
 10  Регион                          39966 non-null  object 
 11  Наименование дилерского центра  39966 non-null  object 
 12  Количество                      

In [7]:
# вполне полноценный датасет получается, а теперь переименуем названия столбцов
df.columns = ['year', 
              'month',
              'brand',
              'model',
              'manufacture_year',
              'manufacture_contry',
              'fuel_type',
              'engine_volume',
              'transmission',
              'drive_type',
              'region',
              'dealer',
              'amount',
              'price_usd',
              'sale_usd',
              'area',
              'segmentation',
              'class']

In [8]:
df = df.drop_duplicates()
df = df.dropna(how = 'all')
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17978 entries, 0 to 39951
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   year                17978 non-null  int64  
 1   month               17978 non-null  object 
 2   brand               17978 non-null  object 
 3   model               17978 non-null  object 
 4   manufacture_year    17779 non-null  object 
 5   manufacture_contry  17978 non-null  object 
 6   fuel_type           16801 non-null  object 
 7   engine_volume       16424 non-null  object 
 8   transmission        16835 non-null  object 
 9   drive_type          16489 non-null  object 
 10  region              17978 non-null  object 
 11  dealer              17978 non-null  object 
 12  amount              17972 non-null  float64
 13  price_usd           17978 non-null  float64
 14  sale_usd            17978 non-null  float64
 15  area                17978 non-null  object 
 16  segmentat

In [9]:
df.head()

,year,month,brand,model,manufacture_year,manufacture_contry,fuel_type,engine_volume,transmission,drive_type,region,dealer,amount,price_usd,sale_usd,area,segmentation,class
0,2019,Май,Audi,A3,2018,Германия,Бензин,"1,4",S-tronic,передний,Алматы,Mercur Auto Алматы,1.0,28115.00000,28115.00000,г.Алматы,Легковые автомобили,C класс
1,2019,Август,Audi,A3,2018,Германия,Бензин,"1,4",S-tronic,передний,Алматы,Mercur Auto Алматы,1.0,32246.99000,32246.99000,г.Алматы,Легковые автомобили,C класс
2,2019,Апрель,Audi,A4,2018,Германия,Бензин,"1,4",S-Tronic,FWD,Алматы,Mercur Auto Алматы,1.0,32000.00000,32000.00000,г.Алматы,Легковые автомобили,D класс
3,2019,Июль,Audi,A4,2018,Германия,Бензин,"1,4",S-tronic,передний,Алматы,Mercur Auto Алматы,1.0,31929.00000,31929.00000,г.Алматы,Легковые автомобили,D класс
5,2019,Февраль,Audi,A6,2017,Германия,Бензин,2,S-Tronic,quattro,Нур-Султан,Mercur Auto Астана,1.0,42608.25485,42608.25485,г.Нур-Султан,Легковые автомобили,E класс


## 2. Глубокая очистка и преобразование данных

In [10]:
# кодируем названия стран в трёхбуквенные коды Alpha3
df['manufacture_contry'].unique()

array(['Германия', 'США', 'Австрия', 'Республика Казахстан',
       'Российская Федерация', 'Корея', 'Япония', 'Таиланд', 'Китай',
       'UK', 'Узбекистан', 'Венгрия', 'Турция', 'Испания', 'Нидерланды',
       'Польша', 'Швеция', 'Белоруссия', 'Бельгия'], dtype=object)

In [11]:
ru_to_iso3 = {
    'Германия':              'DEU',
    'США':                   'USA',
    'Австрия':               'AUT',
    'Республика Казахстан':  'KAZ',
    'Российская Федерация':  'RUS',
    'Корея':                 'KOR',
    'Япония':                'JPN',
    'Таиланд':               'THA',
    'Китай':                 'CHN',
    'Узбекистан':            'UZB',
    'Венгрия':               'HUN',
    'Турция':                'TUR',
    'Испания':               'ESP',
    'Нидерланды':            'NLD',
    'Польша':                'POL',
    'Швеция':                'SWE',
    'Белоруссия':            'BLR',
    'Бельгия':               'BEL',
}

df['manufacture_contry'] = df['manufacture_contry'].map(ru_to_iso3)

df['manufacture_contry'].unique()
#print(df['manufacture_contry'].head())

array(['DEU', 'USA', 'AUT', 'KAZ', 'RUS', 'KOR', 'JPN', 'THA', 'CHN', nan,
       'UZB', 'HUN', 'TUR', 'ESP', 'NLD', 'POL', 'SWE', 'BLR', 'BEL'],
      dtype=object)

In [12]:
df['fuel_type'].unique()

array(['Бензин', 'Дизель', nan, 'дизель ', 'бензин ', 'гибрид', '2',
       '1,6', '0', 'Электро', 'Электричество'], dtype=object)

видим местами откровенный мусор, помимо странных именований есть числа, которые могут сигнализировать об
объёме двигателя, однако однозначно сказать - это дизель, бензин или гибрид -  нельзя, такие данные придётся удалить

In [13]:
# успешно закодировали страны
# обрабатываем столбец, кодируем тип топлива и проверяем результат
# нормализуем строки

df['fuel_type'] = df['fuel_type'].astype('string').str.lower().str.strip()

# пишем маску, которая поможет отсечь числовые значения

numeric_mask = pd.to_numeric(df['fuel_type'].str.replace(',','.', regex = False), errors = 'coerce').notna()

df = df[~numeric_mask].copy()

df['fuel_type'] = df['fuel_type'].replace('электричество','электро')

fuel_type_dict = {
    'бензин':'F',
    'дизель':'D',
    'электро':'E',
    'гибрид':'HYB'
}

df['fuel_type'] = df['fuel_type'].map(fuel_type_dict)


In [14]:
print(df['fuel_type'].unique())
print("Осталось NaN:", df['fuel_type'].isna().sum())

['F' 'D' nan 'HYB' 'E']
Осталось NaN: 1177


In [15]:
# смотрим на типы привода
df['drive_type'].value_counts()

drive_type
4WD              4328
Передний         2883
2 WD             1963
Полный           1674
2WD              1442
4 WD             1020
Задний            932
FF                823
AWD               564
FWD               496
RWD               143
передний           49
quattro            42
0                  31
4Motion            25
Передний (FF)      10
4х2.2              10
4x4                 5
полный              5
4x2                 4
4х4                 4
#Н/Д                3
4х2                 2
4X2                 2
Астана              2
4X4                 2
Name: count, dtype: int64

в целом работать можно, видно что небольшую часть данных придётся отбросить (значения типа #Н/Д или 0),
но без этого никуда
мы разделим всю технику на 3 категории: 4WD, 2WD и RWD

In [16]:
df['drive_type'] = df['drive_type'].str.strip()

# план такой - сначала заменяем то, что можно опознать и классифицировать, затем неклассифицируемые строки удаляем

fwd_list = ['Передний', '2 WD', '2WD', 'FWD', 'передний',
            'Передний (FF)', '4х2.2', '4x2', '4х2', '4X2',
            'FF', 'Передний (FF)']

rwd_list = ['RWD', 'Задний']
four_wd_list = ['Полный', '4 WD', 'AWD', 'quattro',
                '4Motion', 'полный', '4X4', '4х4', '4x4']

df.loc[df['drive_type'].isin(fwd_list), 'drive_type'] = 'FWD'


types_list = [fwd_list, rwd_list, four_wd_list]
codes_list = ['2WD', 'RWD', '4WD']

for types, code in zip(types_list, codes_list):
    df.loc[df['drive_type'].isin(types), 'drive_type'] = code

print(df['drive_type'].value_counts(dropna = False))

drive_type
2WD       7684
4WD       7669
NaN       1489
RWD       1075
0           31
#Н/Д         3
Астана       2
Name: count, dtype: int64


In [17]:
# Значения, которые явно не похожи на код привода и не NaN
mask = ~df['drive_type'].isin({'FWD', 'RWD', '4WD', '2WD', pd.NA})
print(df.loc[mask, 'drive_type'].value_counts(dropna=False))

drive_type
NaN       1489
0           31
#Н/Д         3
Астана       2
Name: count, dtype: int64


In [18]:
valid = {'2WD', 'RWD', '4WD'}
df = df[df['drive_type'].isin(valid)]
df['drive_type'].value_counts(dropna = False)

drive_type
2WD    7684
4WD    7669
RWD    1075
Name: count, dtype: int64

в целом работать можно, теперь займёмся следующей задачей 
"Числовые столбцы: приведите столбцы Стоимость, Количество и Итоговая стоимость 
к корректным числовым типам. Проанализируйте аномалии. Например, Количество = 
-1 это возврат; 100 проданных Skoda — оптовый заказ таксопарка."

In [19]:
df['amount'].value_counts(dropna = False)

amount
 1.0      13878
 2.0        946
 3.0        410
 4.0        259
 5.0        164
 6.0        113
 7.0         96
 8.0         75
 10.0        61
 9.0         59
 12.0        42
 11.0        35
 14.0        32
 13.0        30
 15.0        24
 16.0        20
 18.0        20
 17.0        17
 20.0        14
 24.0        12
 23.0        12
 19.0        12
 21.0        10
 25.0         9
 26.0         9
 22.0         7
 30.0         7
 NaN          6
 32.0         6
 50.0         5
 29.0         4
 27.0         4
 28.0         3
 31.0         3
 33.0         3
 36.0         3
 100.0        1
 41.0         1
 38.0         1
 42.0         1
 51.0         1
 43.0         1
 40.0         1
 34.0         1
 37.0         1
 54.0         1
 63.0         1
 60.0         1
 66.0         1
 35.0         1
 47.0         1
 46.0         1
 115.0        1
-1.0          1
Name: count, dtype: int64

In [20]:
# всего 6 значений NaN - удаляем
df = df.dropna(subset=['amount']).copy()
df['amount'] = df['amount'].astype(int)
df[['price_usd','sale_usd']] = round(df[['price_usd','sale_usd']],2)

In [21]:
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 16422 entries, 0 to 39950
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   year                16422 non-null  int64  
 1   month               16422 non-null  object 
 2   brand               16422 non-null  object 
 3   model               16422 non-null  object 
 4   manufacture_year    16373 non-null  object 
 5   manufacture_contry  16256 non-null  object 
 6   fuel_type           16210 non-null  object 
 7   engine_volume       16226 non-null  object 
 8   transmission        16328 non-null  object 
 9   drive_type          16422 non-null  object 
 10  region              16422 non-null  object 
 11  dealer              16422 non-null  object 
 12  amount              16422 non-null  int64  
 13  price_usd           16422 non-null  float64
 14  sale_usd            16422 non-null  float64
 15  area                16422 non-null  object 
 16  segmentat

,year,month,brand,model,manufacture_year,manufacture_contry,fuel_type,engine_volume,transmission,drive_type,region,dealer,amount,price_usd,sale_usd,area,segmentation,class
0,2019,Май,Audi,A3,2018,DEU,F,"1,4",S-tronic,2WD,Алматы,Mercur Auto Алматы,1,28115.00,28115.00,г.Алматы,Легковые автомобили,C класс
1,2019,Август,Audi,A3,2018,DEU,F,"1,4",S-tronic,2WD,Алматы,Mercur Auto Алматы,1,32246.99,32246.99,г.Алматы,Легковые автомобили,C класс
2,2019,Апрель,Audi,A4,2018,DEU,F,"1,4",S-Tronic,2WD,Алматы,Mercur Auto Алматы,1,32000.00,32000.00,г.Алматы,Легковые автомобили,D класс
3,2019,Июль,Audi,A4,2018,DEU,F,"1,4",S-tronic,2WD,Алматы,Mercur Auto Алматы,1,31929.00,31929.00,г.Алматы,Легковые автомобили,D класс
5,2019,Февраль,Audi,A6,2017,DEU,F,2,S-Tronic,4WD,Нур-Султан,Mercur Auto Астана,1,42608.25,42608.25,г.Нур-Султан,Легковые автомобили,E класс


In [22]:

month_map = {
    'январь': 1,  'февраль': 2,  'март': 3,
    'апрель': 4,  'май': 5,      'июнь': 6,
    'июль': 7,    'август': 8,   'сентябрь': 9,
    'октябрь': 10,'ноябрь': 11,  'декабрь': 12,
}

month_num = (
    df['month'].astype('string')
      .str.strip().str.lower()
      .map(month_map)
      .astype('Int64')
)

year_month = (
    df['year'].astype('string') + '-' +
    month_num.astype('string').str.zfill(2)
)

df['sale_dt'] = (
    pd.to_datetime(year_month, format='%Y-%m', errors='coerce')
    + pd.offsets.MonthEnd(0)
)

df = df.drop(columns = ['year', 'month'])

In [23]:
df.head()

,brand,model,manufacture_year,manufacture_contry,fuel_type,engine_volume,transmission,drive_type,region,dealer,amount,price_usd,sale_usd,area,segmentation,class,sale_dt
0,Audi,A3,2018,DEU,F,"1,4",S-tronic,2WD,Алматы,Mercur Auto Алматы,1,28115.00,28115.00,г.Алматы,Легковые автомобили,C класс,2019-05-31
1,Audi,A3,2018,DEU,F,"1,4",S-tronic,2WD,Алматы,Mercur Auto Алматы,1,32246.99,32246.99,г.Алматы,Легковые автомобили,C класс,2019-08-31
2,Audi,A4,2018,DEU,F,"1,4",S-Tronic,2WD,Алматы,Mercur Auto Алматы,1,32000.00,32000.00,г.Алматы,Легковые автомобили,D класс,2019-04-30
3,Audi,A4,2018,DEU,F,"1,4",S-tronic,2WD,Алматы,Mercur Auto Алматы,1,31929.00,31929.00,г.Алматы,Легковые автомобили,D класс,2019-07-31
5,Audi,A6,2017,DEU,F,2,S-Tronic,4WD,Нур-Султан,Mercur Auto Астана,1,42608.25,42608.25,г.Нур-Султан,Легковые автомобили,E класс,2019-02-28


вроде данные преобразовались, займёмся названиями салона 
"удалите неявные дубликаты, приведя названия дилерских центров 
к единому варианту. Например, «Меркур Авто» и «Mercur Auto» — это один и тот же автоцентр."

In [24]:
df['dealer'].value_counts(dropna = False)

dealer
БИПЭК АВТО Астана          464
Автомир-Центр Караганда    439
Hyundai Premium Almaty     418
Hyundai Premium Astana     411
Hyundai Auto Almaty        409
                          ... 
AST-commerce                 1
Nord MMC                     1
Автогранд Актау              1
Равон Моторс Костанай        1
Ford Атырау                  1
Name: count, Length: 171, dtype: int64

In [25]:
# беда, названий разных слишком много, попробуем первично форматировать их
df['dealer'] = df['dealer'].str.lower().str.strip() 
print(df['dealer'].unique())
# вместо 171 стало 170, успех)

['mercur auto алматы' 'mercur auto астана' 'автоцентр-бавария астана'
 'автоцентр-бавария алматы' 'автоцентр бавария восток'
 'жайык hyundai motors' 'hyundai auto kostanai' 'бипэк авто костанай'
 'бипэк авто астана' 'бипэк авто павлодар' 'бипэк авто алматы'
 'бипэк авто семей' 'бипэк авто шымкент' 'бипэк авто актау'
 'бипэк авто усть-каменогорск' 'бипэк авто атырау' 'бипэк авто актобе'
 'бипэк авто уральск' 'бипэк авто караганда' 'бипэк авто петропавловск'
 'вираж алматы' 'вираж кокшетау' 'вираж уральск' 'вираж петропавловск'
 'вираж павлодар' 'вираж караганда' 'вираж астана' 'вираж костанай'
 'вираж актау' 'вираж усть-каменогорск' 'вираж актобе' 'вираж атырау'
 'вираж шымкент' 'вираж семей' 'вираж рудный' 'вираж кызылорда'
 'вираж тараз' 'cadillac almaty' 'daewoo bus kazakhstan' 'авто сити'
 'ford атырау' 'вираж талдыкорган' 'hyundai auto almaty'
 'hyundai premium almaty' 'hyundai premium astana' 'aktobe hyundai motors'
 'hyundai auto astana' 'astana motor aktau' 'hyundai center pavlo

In [26]:
df['dealer'] = (
    df['dealer'].astype('string')
      .str.strip()
      .str.lower()
      .str.replace('\xa0', ' ', regex=False)    # неразрывный пробел в 'alan motors\xa0...'
      .str.replace('ё', 'е', regex=False)       # на всякий случай
      .str.replace(r'\s+', ' ', regex=True)     # двойные пробелы
)

# через LLM перебираем названия диллерских центров и составляем словарь
dealer_brand_map = {

    # ===== Mercur Auto =====
    'mercur auto алматы':              'mercur auto',
    'mercur auto астана':              'mercur auto',
    'mercur auto атырау':              'mercur auto',

    # ===== Автоцентр-Бавария =====
    'автоцентр-бавария астана':        'автоцентр-бавария',
    'автоцентр-бавария алматы':        'автоцентр-бавария',
    'автоцентр бавария восток':        'автоцентр-бавария',

    # ===== Hyundai (все подвиды) =====
    'жайык hyundai motors':            'hyundai',
    'hyundai auto kostanai':           'hyundai',
    'hyundai auto almaty':             'hyundai',
    'hyundai premium almaty':          'hyundai',
    'hyundai premium astana':          'hyundai',
    'aktobe hyundai motors':           'hyundai',
    'hyundai auto astana':             'hyundai',
    'hyundai center pavlodar':         'hyundai',
    'hyundai center shimkent':         'hyundai',
    'hyundai center kokshetau':        'hyundai',
    'hyundai center petropavlovsk':    'hyundai',
    'hyundai центр жетысу':            'hyundai',
    'hyundai center taraz':            'hyundai',

    # ===== Бипэк Авто =====
    'бипэк авто костанай':             'бипэк авто',
    'бипэк авто астана':               'бипэк авто',
    'бипэк авто павлодар':             'бипэк авто',
    'бипэк авто алматы':               'бипэк авто',
    'бипэк авто семей':                'бипэк авто',
    'бипэк авто шымкент':              'бипэк авто',
    'бипэк авто актау':                'бипэк авто',
    'бипэк авто усть-каменогорск':     'бипэк авто',
    'бипэк авто атырау':               'бипэк авто',
    'бипэк авто актобе':               'бипэк авто',
    'бипэк авто уральск':              'бипэк авто',
    'бипэк авто караганда':            'бипэк авто',
    'бипэк авто петропавловск':        'бипэк авто',
    'бипэк авто кызылорда':            'бипэк авто',
    'бипэк авто кокшетау':             'бипэк авто',
    'бипэк авто тараз':                'бипэк авто',
    'бипэк авто талдыкорган':          'бипэк авто',
    'бипэк авто каскелен':             'бипэк авто',
    'бипэк авто экибастуз':            'бипэк авто',
    'бипэк авто риддер':               'бипэк авто',

    # ===== Вираж =====
    'вираж алматы':                    'вираж',
    'вираж кокшетау':                  'вираж',
    'вираж уральск':                   'вираж',
    'вираж петропавловск':             'вираж',
    'вираж павлодар':                  'вираж',
    'вираж караганда':                 'вираж',
    'вираж астана':                    'вираж',
    'вираж костанай':                  'вираж',
    'вираж актау':                     'вираж',
    'вираж усть-каменогорск':          'вираж',
    'вираж актобе':                    'вираж',
    'вираж атырау':                    'вираж',
    'вираж шымкент':                   'вираж',
    'вираж семей':                     'вираж',
    'вираж рудный':                    'вираж',
    'вираж кызылорда':                 'вираж',
    'вираж тараз':                     'вираж',
    'вираж талдыкорган':               'вираж',

    # ===== Allur Auto =====
    'allur auto almaty':               'allur auto',
    'allur auto astana':               'allur auto',
    'allur auto кызылорда':            'allur auto',
    'allur auto shymkent':             'allur auto',
    'allur auto костанай':             'allur auto',
    'allur auto kokshetau':            'allur auto',
    'allur auto petropavlovsk':        'allur auto',
    'allur auto pavlodar':             'allur auto',
    'allur auto karaganda':            'allur auto',
    'allur auto atyrau':               'allur auto',
    'allur auto ust-kamenogorsk':      'allur auto',
    'allur auto актобе':               'allur auto',
    'allur auto taraz':                'allur auto',

    # ===== Toyota =====
    'toyota motor kazakhstan':         'toyota',
    'toyota city астана':              'toyota',
    'toyota city':                     'toyota',
    'toyota city алматы':              'toyota',
    'тойота центр шымкент':            'toyota',
    'тойота центр шыгыс':              'toyota',
    'тойота центр орал':               'toyota',
    'тойота центр павлодар':           'toyota',
    'тойота центр костанай':           'toyota',
    'тойота центр караганды':          'toyota',
    'тойота центр жетысу':             'toyota',
    'тойота центр атырау':             'toyota',
    'тойота центр астана':             'toyota',
    'тойота центр алматы':             'toyota',
    'тойота центр актобе':             'toyota',
    'тойота центр актау':              'toyota',
    'тойота центр бишкек':             'toyota',

    # ===== Lexus =====
    'лексус астана':                   'lexus',
    'лексус алматы':                   'lexus',
    'лексус актобе':                   'lexus',
    'лексус шымкент':                  'lexus',
    'лексус бишкек':                   'lexus',

    # ===== Subaru =====
    'subaru motor karagandy':          'subaru',
    'subaru motor astana':             'subaru',
    'subaru motor almaty':             'subaru',

    # ===== Mitsubishi =====
    'митсубиши центр астана':          'mitsubishi',
    'mitsubishi centr костанай':       'mitsubishi',

    # ===== Nissan =====
    'nissan centr уральск':            'nissan',
    'nissan center уральск':           'nissan',

    # ===== Ravon =====
    'равон моторс':                    'равон моторс',
    'равон моторс алматы':             'равон моторс',
    'равон моторс сарыагаш':           'равон моторс',
    'равон моторс костанай':           'равон моторс',

    # ===== Кристалл Авто =====
    'кристалл авто кызылорда':         'кристалл авто',
    'кристалл авто караганда':         'кристалл авто',
    'кристалл авто астана':            'кристалл авто',

    # ===== Aster Auto =====
    'aster auto алматы':               'aster auto',
    'aster auto шымкент':              'aster auto',
    'aster auto cемей':                'aster auto',

    # ===== Eurasia Motor =====
    'eurasia motor шымкент':           'eurasia motor',
    'eurasia motor павлодар':          'eurasia motor',
    'eurasia motor атырау':            'eurasia motor',
    'eurasia motor актау':             'eurasia motor',
    'eurasia motor zhaik':             'eurasia motor',
    'eurasia motors uralsk':           'eurasia motor',

    # ===== Автомир-Центр =====
    'автомир-центр караганда':         'автомир-центр',
    'автомир-центр алматы':            'автомир-центр',

    # ===== Сары Арка =====
    'сары арка павлодар':              'сары арка',
    'сары арка':                       'сары арка',

    # ===== Одиночные бренды =====
    'cadillac almaty':                 'cadillac',
    'daewoo bus kazakhstan':           'daewoo',
    'авто сити':                       'авто сити',
    'ford атырау':                     'ford',
    'astana motor aktau':              'astana motor',
    'best-avto':                       'best-avto',
    'caspi auto атырау':               'caspi auto',
    'тенгри-авто':                     'тенгри-авто',
    'кол авто':                        'кол авто',
    'semey motors':                    'semey motors',
    'astana motors':                   'astana motors',
    'терра моторс':                    'терра моторс',
    'cbc алматы':                      'cbc',
    'казбеллизинг':                    'казбеллизинг',
    'автодом костанай':                'автодом',
    'рус-моторс':                      'рус-моторс',
    'камкор-авто павлодар':            'камкор-авто',
    'вивага лада':                     'lada',
    'автоцентр бахус':                 'автоцентр бахус',
    'auto trader':                     'auto trader',
    'автокомплекс тарлан':             'автокомплекс тарлан',
    'автогранд актау':                 'автогранд',
    'sky-авто':                        'sky-авто',
    'nord mmc':                        'nord mmc',
    'тенке капитал':                   'тенке капитал',
    'ast-commerce':                    'ast-commerce',
    'экспорт':                         'экспорт',
    'almaty motors premium':           'almaty motors',
    'урал-кров авто':                  'урал-кров авто',
    'autogood караганда':              'autogood',
    'abs-invest':                      'abs-invest',
    'shym auto':                       'shym auto',
    'blue star of kazakhstan':         'blue star',
    'gold star motors':                'gold star motors',
    'west star motors':                'west star motors',
    'а-моторс авто':                   'а-моторс',
    'автоцентр астэк':                 'автоцентр астэк',
    'lion autocenter almaty':          'lion autocenter',
    'porsche centre almaty':           'porsche',
    'арыстан моторс':                  'арыстан моторс',
    'автоцентр tosa':                  'автоцентр tosa',
    'фаворит моторс':                  'фаворит моторс',
    'жана suzuki':                     'suzuki',
    'alan motors':                     'alan motors',
    'alan motors усть-каменогорск':    'alan motors',   # \xa0 превратится в пробел
    'lex auto':                        'lex auto',
    'autoamir':                        'autoamir',
    'avenue motors':                   'avenue motors',
    'das auto':                        'das auto',
    'volvo car almaty':                'volvo',
    'hino motors kazakhstan':          'hino motors',
}

In [27]:
df['dealer_brand'] = df['dealer'].map(dealer_brand_map).fillna(df['dealer'])
df = df.drop(columns = 'dealer')

In [28]:
df['dealer_brand'].unique()

array(['mercur auto', 'автоцентр-бавария', 'hyundai', 'бипэк авто',
       'вираж', 'cadillac', 'daewoo', 'авто сити', 'ford', 'astana motor',
       'best-avto', 'caspi auto', 'автомир-центр', 'тенгри-авто',
       'кол авто', 'eurasia motor', 'semey motors', 'astana motors',
       'терра моторс', 'cbc', 'казбеллизинг', 'allur auto', 'автодом',
       'рус-моторс', 'камкор-авто', 'lada', 'автоцентр бахус',
       'auto trader', 'автокомплекс тарлан', 'автогранд', 'sky-авто',
       'nord mmc', 'тенке капитал', 'ast-commerce', 'экспорт',
       'almaty motors', 'lexus', 'toyota', 'урал-кров авто', 'autogood',
       'abs-invest', 'shym auto', 'blue star', 'gold star motors',
       'west star motors', 'mitsubishi', 'nissan', 'а-моторс',
       'aster auto', 'сары арка', 'автоцентр астэк', 'lion autocenter',
       'porsche', 'арыстан моторс', 'автоцентр tosa', 'фаворит моторс',
       'suzuki', 'alan motors', 'lex auto', 'autoamir', 'avenue motors',
       'равон моторс', 'кристалл ав

с этой задачей вроде справились тоже, займёмся объёмом двигателя

In [29]:
df['engine_volume'].unique()
# с каждым шагом данные всё дремучее и дремучее

array(['1,4', '2', '3', '4,4', '6,6', '1,5', '1,596', '2,4', '1,7',
       '26,7', '25,7', '24,7', '23,7', '22,7', '21,7', '20,7', '19,7',
       '18,7', '17,7', '16,7', '15,7', '14,7', '13,7', '12,7', '11,7',
       '10,7', '9,7', '8,7', '7,7', '6,7', '5,7', '4,7', '3,7', '2,7',
       '6,2', '1,4 Turbo', '2,5', '1,6', '2,8', nan, '4,3', '4,3,',
       '1,6 MPI', '2,0 MPI', '2,4 GDI', '2,4 MPI', '2,5 CRDI VGT',
       '2,5 CRDI WGT', '3,9', '7,5', '12,3', '1,6 T-GDI', '2,0 CRDI',
       '2.0', '3,5', '5,6', '5,2', '3,0 L', '1.6', '1.5', '5', '1,598',
       '1,248', '1,998', '2,359', '1,999', '3,342', '1,591', '3,47',
       '1,69', '1,774', '2.5', '2.7 ', '3.5', '2.7', '4.6', '4,6', '5.7',
       '1,8', '4', '5,5', '12', '12,8', '2,2', '1,2', '1,485', '1,33',
       '4.0', '2.4G', '2,4G', '2.8', '2,693', '2,0', '3,8', '3.8', '1,2T',
       '3,6', '7,6', '4,9', '2.0h', '2,9', '400 Л.С.', '4,98 L,', '4,98',
       '88 KWH', '8,4 L,', '6,7L', '6,5'], dtype=object)

In [30]:

def clean_engine_volume(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(',', '.').lower()

    # 1) отсекаем явный не-объём
    if any(tok in s for tok in ['kwh', 'квт', 'л.с', 'hp', 'kw']):
        return np.nan

    # 2) извлекаем первое число
    m = re.search(r'\d+(?:\.\d+)?', s)
    if not m:
        return np.nan
    v = float(m.group())

    # 3) нормализация масштаба
    if v >= 1000:
        v = v / 1000          # 1596 -> 1.596
    elif v >= 100:
        return np.nan         # 400 Л.С. и т.п.
    elif v >= 10:
        v = v / 10            # 26.7 -> 2.67

    # 4) фильтр по реальному диапазону
    if 0.6 <= v <= 8.0:
        return round(v, 2)
    return np.nan


df['engine_volume'] = df['engine_volume'].apply(clean_engine_volume)
df = df.dropna(subset = ['engine_volume']).copy()

In [31]:
df['engine_volume'].unique()

array([1.4 , 2.  , 3.  , 4.4 , 6.6 , 1.5 , 1.6 , 2.4 , 1.7 , 2.67, 2.57,
       2.47, 2.37, 2.27, 2.17, 2.07, 1.97, 1.87, 1.77, 1.67, 1.57, 1.47,
       1.37, 1.27, 1.17, 1.07, 7.7 , 6.7 , 5.7 , 4.7 , 3.7 , 2.7 , 6.2 ,
       2.5 , 2.8 , 4.3 , 3.9 , 7.5 , 1.23, 3.5 , 5.6 , 5.2 , 5.  , 1.25,
       2.36, 3.34, 1.59, 3.47, 1.69, 4.6 , 1.8 , 4.  , 5.5 , 1.2 , 1.28,
       2.2 , 1.49, 1.33, 2.69, 3.8 , 3.6 , 7.6 , 4.9 , 2.9 , 4.98, 6.5 ])

In [32]:
df['region'].unique()

array(['Алматы', 'Нур-Султан', 'Усть-Каменогорск', 'Уральск', 'Костанай',
       'Павлодар', 'Семей', 'Шымкент', 'Актау', 'Атырау', 'Актобе',
       'Караганда', 'Петропавловск', 'Кокшетау', 'Рудный', 'Кызылорда',
       'Тараз', 'Туркестан', 'Талдыкорган', 'ЭКСПОРТ', 'Каскелен',
       'Экибастуз', 'Риддер', 'Сарыагаш'], dtype=object)

In [33]:
df['area'].unique()

array(['г.Алматы', 'г.Нур-Султан', 'Восточно-Казахстанская область',
       'Западно-Казахстанская область', 'Костанайская область',
       'Павлодарская область', 'Южно-Казахстанская область',
       'Мангистауская область', 'Атырауская область',
       'Актюбинская область', 'Карагандинская область',
       'Северо-Казахстанская область', 'Акмолинская область',
       'Кызылординская область', 'Жамбылская область',
       'Туркестанская область', 'Алматинская область', 'Экспорт область'],
      dtype=object)

In [34]:
df['region'] = df['region'].str.title()
df['area']   = df['area'].str.title()

mask = (
    df['region'].astype('string').str.contains('экспорт', case=False, na=False)
    | df['area'].astype('string').str.contains('экспорт', case=False, na=False)
)
df = df[~mask].copy()

print("Осталось строк:", len(df))
print(df['region'].unique())
print(df['area'].unique())

Осталось строк: 15977
['Алматы' 'Нур-Султан' 'Усть-Каменогорск' 'Уральск' 'Костанай' 'Павлодар'
 'Семей' 'Шымкент' 'Актау' 'Атырау' 'Актобе' 'Караганда' 'Петропавловск'
 'Кокшетау' 'Рудный' 'Кызылорда' 'Тараз' 'Туркестан' 'Талдыкорган'
 'Каскелен' 'Экибастуз' 'Риддер' 'Сарыагаш']
['Г.Алматы' 'Г.Нур-Султан' 'Восточно-Казахстанская Область'
 'Западно-Казахстанская Область' 'Костанайская Область'
 'Павлодарская Область' 'Южно-Казахстанская Область'
 'Мангистауская Область' 'Атырауская Область' 'Актюбинская Область'
 'Карагандинская Область' 'Северо-Казахстанская Область'
 'Акмолинская Область' 'Кызылординская Область' 'Жамбылская Область'
 'Туркестанская Область' 'Алматинская Область']


работаем с типом трансмиссии, категоризируем до автомата и механики

In [35]:
df['transmission'].unique()

array(['S-tronic', 'S-Tronic', 'Tiptronic', 'АКПП', 'АКП', 'Steptronic',
       'AT', '6 АТ', '5 МТ', '6АТ', 'MT', '8AT', '5МТ', 'Powershift S6',
       '6AT', 'Мех.', '6 AT', '6 MT', '6MT', '5 AT', '5AT', 'МКПП',
       '7 DCT', '7DCT', nan, 'МТ', 'CVT', 'DCT', '4 АТ', '6 МТ', '6МТ',
       '4АТ', '4AT', '8АТ', '8 АТ', '5MT', 'AMT', 'CVT (вариатор)',
       'CVT (вариATор)', 'CVT(вAриATор)', '7G-TRONIC', '9G-TRONIC',
       '12AT', '12АТ', '5АТ', '7АТ', '7AT', '6M/T', 'PDK', '8-ступ АКПП',
       '8', '8АКПП', '8 АКПП Tiptronic S', '7 АКПП (PDK)', '7 АКПП PDK',
       '8 АКПП (PDK)', 'АТ', 'АT', 'МT', '7 DSG', '7DSG', '6 DSG', '6DSG',
       'DSG', '6A', '6А', '5M', 'A/T', 'M/T', '5М', 'М/T', ' 7 АКП',
       '7АКП', 'А/T', '8A', 'TDI', '9AT'], dtype=object)

In [36]:


df['transmission'] = df['transmission'].astype('string').str.strip().str.lower()


transmission_map = {
    # ================ АВТОМАТ ================
    's-tronic': 'AT', 's-tronic': 'AT', 'tiptronic': 'AT',
    'steptronic': 'AT', 'powershift s6': 'AT',
    'акпп': 'AT', 'акп': 'AT',
    'at': 'AT', '6 ат': 'AT', '6ат': 'AT', '6at': 'AT', '6 at': 'AT',
    '5 at': 'AT', '5at': 'AT', '5 ат': 'AT', '5ат': 'AT',
    '4 ат': 'AT', '4ат': 'AT', '4at': 'AT',
    '8at': 'AT', '8ат': 'AT', '8 ат': 'AT',
    '7ат': 'AT', '7at': 'AT',
    '12at': 'AT', '12ат': 'AT',
    '9at': 'AT',
    '8-ступ акпп': 'AT', '8акпп': 'AT', '8 акпп tiptronic s': 'AT',
    '7 акпп (pdk)': 'AT', '7 акпп pdk': 'AT', '8 акпп (pdk)': 'AT',
    '7 акп': 'AT', '7акп': 'AT',
    'cvt': 'AT', 'cvt (вариатор)': 'AT', 'cvt (вариatор)': 'AT',
    'cvt(вaриatор)': 'AT',
    'dct': 'AT', '7 dct': 'AT', '7dct': 'AT',
    'dsg': 'AT', '7 dsg': 'AT', '7dsg': 'AT', '6 dsg': 'AT', '6dsg': 'AT',
    'pdk': 'AT',
    'amt': 'AT',
    '7g-tronic': 'AT', '9g-tronic': 'AT',
    'аt': 'AT', 'ат': 'AT',
    'a/t': 'AT', 'а/t': 'AT',
    '6a': 'AT', '6а': 'AT', '8a': 'AT',
    
    # ================ МЕХАНИКА ================
    'mt': 'MT', 'мт': 'MT',
    '5 мт': 'MT', '5мт': 'MT', '5mt': 'MT',
    '6 mt': 'MT', '6mt': 'MT', '6 мт': 'MT', '6мт': 'MT',
    '6m/t': 'MT',
    'm/t': 'MT', 'м/t': 'MT',
    'мкпп': 'MT',
    'мех.': 'MT', 'мt':'MT',
    '5m': 'MT', '5м': 'MT',
}

# 3) маппинг + категориальный тип
df['transmission_type'] = (
    df['transmission'].map(transmission_map).astype('category')
)

# 4) диагностика: что не смапилось
unmapped = df.loc[df['transmission_type'].isna(), 'transmission'].dropna().unique()
print(f"Не распознано ({len(unmapped)}):", sorted(unmapped.tolist()))

# 5) удаляем исходный столбец
df = df.drop(columns=['transmission'])

# 6) результат
print(df['transmission_type'].value_counts(dropna=False))
print("Тип:", df['transmission_type'].dtype)

Не распознано (2): ['8', 'tdi']
transmission_type
AT     11681
MT      4265
NaN       31
Name: count, dtype: int64
Тип: category


In [38]:
# '8' это Porsche Cayenne с 8-ступенчатым автоматом
df.loc[df['transmission_type'] == '8', 'transmission_type'] = 'AT'


df = df[df['transmission_type'].isin(['AT', 'MT'])].copy()


df['transmission_type'] = df['transmission_type'].astype('category')

print(df['transmission_type'].value_counts(dropna=False))
print("Пропусков:", df['transmission_type'].isna().sum())

transmission_type
AT    11681
MT     4265
Name: count, dtype: int64
Пропусков: 0


In [41]:
df.rename(columns={'transmission_type' : 'transmission'}, inplace = True)

df['transmission'] = df['transmission'].cat.rename_categories({
    'AT': 'Автомат',
    'MT': 'Механика',
})

In [44]:
df.columns

Index(['brand', 'model', 'manufacture_year', 'manufacture_contry', 'fuel_type',
       'engine_volume', 'drive_type', 'region', 'amount', 'price_usd',
       'sale_usd', 'area', 'segmentation', 'class', 'sale_dt', 'dealer_brand',
       'transmission'],
      dtype='object')

In [45]:

df['sale_dt'] = pd.to_datetime(df['sale_dt'], errors='coerce')


cat_cols = ['fuel_type', 'transmission', 'drive_type', 'segmentation', 'class']
for col in cat_cols:
    df[col] = df[col].astype('category')

print(df.dtypes)

brand                         object
model                         object
manufacture_year              object
manufacture_contry            object
fuel_type                   category
engine_volume                float64
drive_type                  category
region                        object
amount                         int64
price_usd                    float64
sale_usd                     float64
area                          object
segmentation                category
class                       category
sale_dt               datetime64[ns]
dealer_brand                  object
transmission                category
dtype: object
